## Update Sample Points

### Authors: Mike Bollinger and Madeleine Gallop

Project: BOEM - Carolina Long Bay

May 31, 2024

Summary: finds two existing feature services (one for stratified and one for regular) in ArcGIS Online (created from Karina's ArcDesktop layers), brings in Survey123 data on a daily basis after surveying (the survey details which sites have been visited) and joins the Survey123 data to the feature services. It then concatonates the two feature services into a single CSV, which will go into the dashboard through the BOEM_Dashboard_Update.ipynb 

UPDATED v1 06.14.2024
This version is being updated by Madeleine Gallop 

In [1]:
# import packages
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection
import pandas as pd

# Connect to ArcGIS Online
gis = GIS("home")
print("Logged in to " + str(gis.properties.name) + " as " + str(gis.properties.user.username))

#Existing Feature Service IDs
StratifiedFS = '08a117dcc99b42cf8a7cdf7532a1b891' #feature layer in arc
RegularFS = '8710ded0bb93462f9875ff1dc1547073 ' #feature layer in arc



Logged in to NOAA GeoPlatform as karina.urquhart_noaa
08a117dcc99b42cf8a7cdf7532a1b891


In [2]:
# Path to data from Survey123 documenting new sites sampled
new_data_dbf = pd.read_csv(r"Karina_Madeleine_Test/NF2406_EM2040_DN164_StratSamp.csv") #THIS NEEDS TO BE A PATH TO THE SURVEY123 CSV that we download daily. 
new_data_dbf.shape

# Make sure that the columns are consistent between the two services - is this easier to do in QGIS? 

new_columns = ['AUV_Completed_Date', 'ROV_Completed_Date','DropCam_Completed_Date','Status','Most_Recent_Survey_Date']

# pad new cols with empty string (NaT)
for col in new_columns:
    new_data_dbf[col] = pd.NaT
new_data_dbf.head()


,SiteID,Lon,Lat,Easting,Northing,Design,DateCreated,SurveyDate,Strata,Depth,...,RoughnessThresh,SlopeThresh,WindowSize,AssignedVehicle,PlotterName,AUV_Completed_Date,ROV_Completed_Date,DropCam_Completed_Date,Status,Most_Recent_Survey_Date
0,1,-77.875487,33.476275,790357.5,3708458.5,Stratified,164.0,164.0,2.0,-28.819548,...,0.032,2.0,7.0,DC,DC_1,NaT,NaT,NaT,NaT,NaT
1,2,-77.877906,33.495186,790069.5,3710549.5,Stratified,164.0,164.0,1.0,-27.744108,...,0.032,2.0,7.0,DC,DC_2,NaT,NaT,NaT,NaT,NaT
2,3,-77.879960,33.520370,789794.5,3713337.5,Stratified,164.0,164.0,2.0,-26.545879,...,0.032,2.0,7.0,AUV,AUV_3,NaT,NaT,NaT,NaT,NaT
3,4,-77.875497,33.472093,790370.5,3707994.5,Stratified,164.0,164.0,1.0,-29.019596,...,0.032,2.0,7.0,AUV,AUV_4,NaT,NaT,NaT,NaT,NaT
4,5,-77.882433,33.516240,789578.5,3712872.5,Stratified,164.0,164.0,1.0,-26.276503,...,0.032,2.0,7.0,AUV,AUV_5,NaT,NaT,NaT,NaT,NaT


In [3]:
# Write a path to save the csv that will be uploaded to the dashboard after the join with survey123 points
JoinedCSVpath = r"Karina_Madeleine_Test/Joined_Sampling_Points_Test.csv"


In [3]:
# Assume feature service is already created, make an object in the code for them

StratifiedLayer = gis.content.get(StratifiedFS).layers[0] #0 because it is first and only layer
RegularLayer = gis.content.get(RegularFS).layers[0]

# Make the content into a dataframe

stratified_dbf = pd.DataFrame(StratifiedLayer.query().sdf)
regular_dbf = pd.DataFrame(RegularLayer.query().sdf)

# concatonate the two layers - will be ready for comparison with survey123 data
concatenated_df = pd.concat([stratified_dbf, regular_dbf], ignore_index=True) # I am not sure I want this to be true - experiment with it tomorrow

concatenated_df.head()

,fid,SiteID,Lon,Lat,Easting,Northing,Design,DateCreated,SurveyDate,Strata,...,probabilities,StrataWeights,BackscatterThresh,RoughnessThresh,SlopeThresh,WindowSize,AssignedVehicle,PlotterName,CollectionStatus,SHAPE
0,1,1,-77.875487,33.476275,790357.5,3708458.5,Stratified,164.0,164.0,2.0,...,0.000001,0.75,-15.853,0.032,2.0,7.0,DC,DC_1,<NA>,"{""x"": -8669059.77882549, ""y"": 3958694.43764536..."
1,2,2,-77.877906,33.495186,790069.5,3710549.5,Stratified,164.0,164.0,1.0,...,0.0,0.25,-15.853,0.032,2.0,7.0,DC,DC_2,<NA>,"{""x"": -8669329.044675099, ""y"": 3961218.4264622..."
2,3,3,-77.87996,33.52037,789794.5,3713337.5,Stratified,164.0,164.0,2.0,...,0.000001,0.75,-15.853,0.032,2.0,7.0,AUV,AUV_3,Complete,"{""x"": -8669557.721736273, ""y"": 3964580.6501685..."
3,4,4,-77.875497,33.472093,790370.5,3707994.5,Stratified,164.0,164.0,1.0,...,0.0,0.25,-15.853,0.032,2.0,7.0,AUV,AUV_4,<NA>,"{""x"": -8669060.93959351, ""y"": 3958136.27228816..."
4,5,5,-77.882433,33.51624,789578.5,3712872.5,Stratified,164.0,164.0,1.0,...,0.0,0.25,-15.853,0.032,2.0,7.0,AUV,AUV_5,<NA>,"{""x"": -8669833.005475866, ""y"": 3964029.2847572..."


In [5]:
# join the survey123 data to the concatonated layer, making sure to keep the geometry from the survey123 data
s123_cruise = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[0] #NCCOS UMS database cruise level layer, point
s123_auv = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[1] #NCCOS UMS AUV database point layer
s123_rov = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[2] #NCCOS UMS ROV database point layer
s123_dc = gis.content.get('00d6b4deec944f3992c7a6ce016f3a7d').layers[3] #NCCOS UMS Drop Camera database point layer

# convert to databases
s123_cruise_dbf = pd.DataFrame(s123_cruise.query().sdf)
s123_auv_dbf = pd.DataFrame(s123_auv.query().sdf)
s123_rov_dbf = pd.DataFrame(s123_rov.query().sdf)
s123_dc_dbf = pd.DataFrame(s123_dc.query().sdf)

#concatonate 

concatenated_123_df = pd.concat([s123_cruise_dbf, s123_auv_dbf, s123_rov_dbf, s123_dc_dbf], ignore_index=True) # I am not sure I wanna concatonate here

# save the joined survey data to the JoinedCSVpath, prepare to update the dashboard

merged_df = pd.merge(concatenated_df, concatenated_123_df, on='site_id', how='left') #left joins preserve all rows from 'concatenated_df' and matching rows from 'concatonated_123_df'. 
# experiment with using left right and other

#make sure we are keeping the geometry from concatenated_123_df
##something with updating the shape only for indexes where we have survey123 data.

#fix other parameters - be thoughtful about how you set up the parameters in arc ahead of time and have survey123 do a better job of accounting for different variables

#save to joined csv you set up before 

#update the map on the dashboard -- perhaps complex
